Forward and Backward Selection

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import train_test_split

In [3]:
#Import smallest dataset to be appended to other datasets
#This dataset is measure of a few simple totals that work as a representation of the logistical complexity of each airport
domestic_data_2024 = pd.read_csv(r"C:\Users\lemrd\Downloads\OMDS\Mod B\Sem 2\Data\T100_Domestic_Market_and_Segment_Data_8942359590531559889.csv")
domestic_data_2024.drop(["year", "enplanements", "arrivals", "OBJECTID"], axis=1, inplace=True) #Redundant with other columns
domestic_data_2024.head()

,origin,passengers,departures,freight,mail
0,01A,17,5,0,0
1,05A,1,1,0,0
2,06A,55,67,139,0
3,09A,43,15,0,0
4,1B1,32,7,0,0


In [4]:
#Import first dataset, 2015 Flight Delay Data
#Columns 7,8 needed dtype specified directly as infer failed

flights_2015 = pd.read_csv(r"C:\Users\lemrd\Downloads\OMDS\Mod B\Sem 2\Data\2015_Delay_Data\flights.csv", dtype={"DESTINATION_AIRPORT":str, "ORIGIN_AIRPORT":str})

flights_2015.fillna({"AIR_SYSTEM_DELAY":0,"SECURITY_DELAY":0,"AIRLINE_DELAY":0,"LATE_AIRCRAFT_DELAY":0,"WEATHER_DELAY":0}, inplace=True)
flights_2015.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0,0,NaN,0.0,0.0,0.0,0.0,0.0
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0,0,NaN,0.0,0.0,0.0,0.0,0.0
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0,0,NaN,0.0,0.0,0.0,0.0,0.0
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0,0,NaN,0.0,0.0,0.0,0.0,0.0
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0,0,NaN,0.0,0.0,0.0,0.0,0.0


In [5]:
#Append airport logistics numbers to the 2015 Flight Delay Dataset
domestic_data_2024.rename({"origin" : "ORIGIN_AIRPORT"}, inplace=True, axis=1)
flight_2015_extended = pd.merge(flights_2015, domestic_data_2024, how='left', on="ORIGIN_AIRPORT")
flight_2015_extended.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,passengers,departures,freight,mail
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,NaN,0.0,0.0,0.0,0.0,0.0,2702278.0,71765.0,3.116064e+09,95095127.0
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,NaN,0.0,0.0,0.0,0.0,0.0,26340206.0,206637.0,8.440161e+08,47992961.0
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,NaN,0.0,0.0,0.0,0.0,0.0,17666714.0,138571.0,2.182348e+08,8795256.0
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,NaN,0.0,0.0,0.0,0.0,0.0,26340206.0,206637.0,8.440161e+08,47992961.0
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,NaN,0.0,0.0,0.0,0.0,0.0,22288303.0,191107.0,3.490318e+08,14452644.0


In [6]:
#To perform any linear regression we need to drop non-numeric columns, columns that breakdown the delay into reasons, and unhelpful columns such as Year
#The non-numeric columns need dropped due to the number of unique values and the lack of meaningful directionality of those values
#Also need to drop Elapsed Time which is Air Time plus Taxiing time at both airports as they directly calculate the delay

flight_2015_delay_vars = flight_2015_extended.drop(["YEAR", "AIRLINE", "FLIGHT_NUMBER", "TAIL_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DIVERTED", "ELAPSED_TIME", "AIR_TIME", "TAXI_IN", "TAXI_OUT",
                                                          "CANCELLED", "CANCELLATION_REASON", "AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", 
                                                          "WEATHER_DELAY"], axis=1)
flight_2015_delay_vars.dropna(axis=0, inplace=True)

target = flight_2015_delay_vars["ARRIVAL_DELAY"]
variables = flight_2015_delay_vars.drop("ARRIVAL_DELAY", axis=1)

In [6]:
#Scale features
scaler = StandardScaler()
variables_scl = scaler.fit_transform(variables)

#Train test split
X_train, X_test, y_train, y_test = train_test_split(variables_scl, target, random_state=42)

In [9]:
model = LinearRegression()
#Forward
fwd_sel = SequentialFeatureSelector(model, direction="forward").fit(X_train, y_train)
var_fwd = fwd_sel.transform(X_train)
test_var_fwd = fwd_sel.transform(X_test)

#Backward
bwd_sel = SequentialFeatureSelector(model, direction="backward").fit(X_train, y_train)
var_bwd = bwd_sel.transform(X_train)
test_var_bwd = bwd_sel.transform(X_test)

#Utilize Selected Features
fwd_model = LinearRegression().fit(var_fwd,y_train)
bwd_model = LinearRegression().fit(var_bwd,y_train)

fwd_train_pred = fwd_model.predict(var_fwd)
bwd_train_pred = bwd_model.predict(var_bwd)

fwd_test_pred = fwd_model.predict(test_var_fwd)
bwd_test_pred = bwd_model.predict(test_var_bwd)

train_rmse_fwd = metrics.root_mean_squared_error(y_train,fwd_train_pred)
test_rmse_fwd = metrics.root_mean_squared_error(y_test,fwd_test_pred)
train_rmse_bwd = metrics.root_mean_squared_error(y_train,bwd_train_pred)
test_rmse_bwd = metrics.root_mean_squared_error(y_test,bwd_test_pred)

print(f"The RMSE on the forward selected variables training data was {train_rmse_fwd:.4f} minutes and on the test data was {test_rmse_fwd:.4f} minutes")
print(f"The RMSE on the backward selected variables training data was {train_rmse_bwd:.4f} minutes and on the test data was {test_rmse_bwd:.4f} minutes")

The RMSE on the forward selected variables training data was 12.6605 minutes and on the test data was 12.7160 minutes
The RMSE on the backward selected variables training data was 12.6368 minutes and on the test data was 12.6913 minutes


In [13]:
var_fwd.shape

(3921488, 8)

In [14]:
var_bwd.shape

(3921488, 8)

In [15]:
X_train.shape

(3921488, 16)

The model proved to be just about as accurate as Linear Regression on the full dataset. These models both only use half of the available variables to get that result.

In [2]:
#Import 3rd Dataset

delay_causes = pd.read_csv(r"C:\Users\lemrd\Downloads\OMDS\Mod B\Sem 2\Data\Flight_Delay_and_Causes_Data\Flight_delay.csv")
#Remove the few duplicate datapoints
delay_causes.drop_duplicates(keep="first", inplace=True)
#Drop columns with no information (all values are the same)
delay_causes.drop(["Cancelled","Diverted","CancellationCode"],axis=1, inplace=True)
#Fix long delays to break out of 24 hour time to show the real delay length
data_to_shift = delay_causes[delay_causes["ArrTime"] < (delay_causes["CRSArrTime"] - 100)].index
delay_causes.loc[data_to_shift,"ArrTime"] = delay_causes.loc[data_to_shift,"ArrTime"] + 2400
                           
delay_causes.head()

,DayOfWeek,Date,DepTime,ArrTime,CRSArrTime,UniqueCarrier,Airline,FlightNum,TailNum,ActualElapsedTime,...,Dest,Dest_Airport,Distance,TaxiIn,TaxiOut,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,4,03-01-2019,1829,1959,1925,WN,Southwest Airlines Co.,3920,N464WN,90,...,BWI,Baltimore-Washington International Airport,515,3,10,2,0,0,0,32
1,4,03-01-2019,1937,2037,1940,WN,Southwest Airlines Co.,509,N763SW,240,...,LAS,McCarran International Airport,1591,3,7,10,0,0,0,47
2,4,03-01-2019,1644,1845,1725,WN,Southwest Airlines Co.,1333,N334SW,121,...,MCO,Orlando International Airport,828,6,8,8,0,0,0,72
3,4,03-01-2019,1452,1640,1625,WN,Southwest Airlines Co.,675,N286WN,228,...,PHX,Phoenix Sky Harbor International Airport,1489,7,8,3,0,0,0,12
4,4,03-01-2019,1323,1526,1510,WN,Southwest Airlines Co.,4,N674AA,123,...,TPA,Tampa International Airport,838,4,9,0,0,0,0,16


In [7]:
#Append Logistics Dataset
domestic_data_2024.rename({"ORIGIN_AIRPORT" : "Origin"}, inplace=True, axis=1)
delay_causes_extended = pd.merge(delay_causes, domestic_data_2024, how='left', on="Origin")
delay_causes_extended.head()

,DayOfWeek,Date,DepTime,ArrTime,CRSArrTime,UniqueCarrier,Airline,FlightNum,TailNum,ActualElapsedTime,...,TaxiOut,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,passengers,departures,freight,mail
0,4,03-01-2019,1829,1959,1925,WN,Southwest Airlines Co.,3920,N464WN,90,...,10,2,0,0,0,32,5194000.0,60772.0,842478043.0,84741.0
1,4,03-01-2019,1937,2037,1940,WN,Southwest Airlines Co.,509,N763SW,240,...,7,10,0,0,0,47,5194000.0,60772.0,842478043.0,84741.0
2,4,03-01-2019,1644,1845,1725,WN,Southwest Airlines Co.,1333,N334SW,121,...,8,8,0,0,0,72,5194000.0,60772.0,842478043.0,84741.0
3,4,03-01-2019,1452,1640,1625,WN,Southwest Airlines Co.,675,N286WN,228,...,8,3,0,0,0,12,5194000.0,60772.0,842478043.0,84741.0
4,4,03-01-2019,1323,1526,1510,WN,Southwest Airlines Co.,4,N674AA,123,...,9,0,0,0,0,16,5194000.0,60772.0,842478043.0,84741.0


In [8]:
#Drop non-numeric columns
num_delay_causes = delay_causes_extended.drop(["Date", "UniqueCarrier", "Airline", "TailNum", "Origin", "Org_Airport", "Dest", "Dest_Airport"], axis = 1)
#Drop missing data
num_delay_causes.dropna(axis=0, inplace=True)

target_delay = num_delay_causes["ArrDelay"]
variables_delay = num_delay_causes.drop(["ArrDelay","CarrierDelay", "WeatherDelay",  "NASDelay", "SecurityDelay", "LateAircraftDelay",
                                         "AirTime", "ArrTime","ActualElapsedTime"], axis = 1)

In [9]:
#Scale features
scaler = StandardScaler()
variables_scl_delay = scaler.fit_transform(variables_delay)

#Train test split
X_train, X_test, y_train, y_test = train_test_split(variables_scl_delay, target_delay, random_state=42)

In [10]:
model = LinearRegression()
#Forward
fwd_sel = SequentialFeatureSelector(model, direction="forward").fit(X_train, y_train)
var_fwd = fwd_sel.transform(X_train)
test_var_fwd = fwd_sel.transform(X_test)

#Backward
bwd_sel = SequentialFeatureSelector(model, direction="backward").fit(X_train, y_train)
var_bwd = bwd_sel.transform(X_train)
test_var_bwd = bwd_sel.transform(X_test)

#Utilize Selected Features
fwd_model = LinearRegression().fit(var_fwd,y_train)
bwd_model = LinearRegression().fit(var_bwd,y_train)

fwd_train_pred = fwd_model.predict(var_fwd)
bwd_train_pred = bwd_model.predict(var_bwd)

fwd_test_pred = fwd_model.predict(test_var_fwd)
bwd_test_pred = bwd_model.predict(test_var_bwd)

train_rmse_fwd = metrics.root_mean_squared_error(y_train,fwd_train_pred)
test_rmse_fwd = metrics.root_mean_squared_error(y_test,fwd_test_pred)
train_rmse_bwd = metrics.root_mean_squared_error(y_train,bwd_train_pred)
test_rmse_bwd = metrics.root_mean_squared_error(y_test,bwd_test_pred)

print(f"The RMSE on the forward selected variables training data was {train_rmse_fwd:.4f} minutes and on the test data was {test_rmse_fwd:.4f} minutes")
print(f"The RMSE on the backward selected variables training data was {train_rmse_bwd:.4f} minutes and on the test data was {test_rmse_bwd:.4f} minutes")

The RMSE on the forward selected variables training data was 10.7400 minutes and on the test data was 10.7198 minutes
The RMSE on the backward selected variables training data was 10.7306 minutes and on the test data was 10.7074 minutes


In [11]:
var_fwd.shape

(363184, 6)

In [12]:
var_bwd.shape

(363184, 7)

In [13]:
X_train.shape

(363184, 13)